# How Embeddings Actually Work

Three questions this notebook answers:
1. How can a single word like 'cat' become a vector? There's nothing to compare it to.
2. Do we always need negative pairs for training?
3. What's the difference between an embedding model and a vector DB?

---
## Question 1: How does 'cat' become a vector?

Your intuition: "Vectors are relative. A single word can't have a vector because there's nothing to compare."

You're right that vectors are LEARNED from comparisons. But those comparisons happened
**during training**, not at query time. The model saw billions of sentences:

```
"The cat sat on the mat"          -> cat appears near: sat, mat
"I fed my cat this morning"       -> cat appears near: fed, morning
"Dogs and cats are common pets"   -> cat appears near: dogs, pets
"The cat chased a mouse"          -> cat appears near: chased, mouse
```

After training, 'cat' has a fixed position in vector space.
When you call `embed('cat')`, it's just a lookup — no comparison needed.

Let's see this happen step by step.

In [ ]:
import numpy as np
np.random.seed(42)

# BEFORE training: each word gets a RANDOM vector
# These mean nothing yet
vocab = ['cat', 'dog', 'pet', 'fish', 'car', 'engine', 'wheel', 'drive']
embed_dim = 4
embeddings = {word: np.random.randn(embed_dim) * 0.5 for word in vocab}

def cosine(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8)

print("BEFORE training — random vectors, no meaning:\n")
print(f"  cat = [{', '.join(f'{v:.2f}' for v in embeddings['cat'])}]")
print(f"  dog = [{', '.join(f'{v:.2f}' for v in embeddings['dog'])}]")
print(f"  car = [{', '.join(f'{v:.2f}' for v in embeddings['car'])}]")
print()
print(f"  cosine(cat, dog) = {cosine(embeddings['cat'], embeddings['dog']):.3f}  (random)")
print(f"  cosine(cat, car) = {cosine(embeddings['cat'], embeddings['car']):.3f}  (random)")
print()
print("  These numbers are meaningless. cat and car might be closer than cat and dog.")
print("  The model hasn't seen any text yet.")

In [ ]:
# Simulate what happens during training
sentences = [
    ['cat', 'pet', 'fish'],
    ['dog', 'pet', 'fish'],
    ['cat', 'dog', 'pet'],
    ['car', 'engine', 'drive'],
    ['car', 'wheel', 'drive'],
    ['engine', 'wheel', 'car'],
]

lr = 0.3
for epoch in range(200):
    for sentence in sentences:
        for i in range(len(sentence)):
            for j in range(len(sentence)):
                if i != j:
                    w1, w2 = sentence[i], sentence[j]
                    diff = embeddings[w2] - embeddings[w1]
                    embeddings[w1] += lr * 0.01 * diff
        for w_in in sentence:
            for w_out in vocab:
                if w_out not in sentence:
                    diff = embeddings[w_out] - embeddings[w_in]
                    embeddings[w_in] -= lr * 0.005 * diff

print("AFTER training — vectors now encode meaning:\n")
print(f"  cat = [{', '.join(f'{v:.2f}' for v in embeddings['cat'])}]")
print(f"  dog = [{', '.join(f'{v:.2f}' for v in embeddings['dog'])}]")
print(f"  car = [{', '.join(f'{v:.2f}' for v in embeddings['car'])}]")
print()
print(f"  cosine(cat, dog) = {cosine(embeddings['cat'], embeddings['dog']):.3f}  <- similar (both pets)")
print(f"  cosine(cat, pet) = {cosine(embeddings['cat'], embeddings['pet']):.3f}  <- similar (cat is a pet)")
print(f"  cosine(cat, car) = {cosine(embeddings['cat'], embeddings['car']):.3f}  <- different")
print(f"  cosine(car, engine) = {cosine(embeddings['car'], embeddings['engine']):.3f}  <- similar (car parts)")
print()
print("  NOW when you ask for embed('cat'), it returns a meaningful vector.")
print("  The comparisons already happened during training.")
print("  The vector is a SUMMARY of everything the model learned about 'cat'.")

In [ ]:
print("What do the vector dimensions represent?\n")

print(f"  {'Word':<8} {'dim0':>6} {'dim1':>6} {'dim2':>6} {'dim3':>6}")
print(f"  {'----':<8} {'----':>6} {'----':>6} {'----':>6} {'----':>6}")
for word in vocab:
    v = embeddings[word]
    print(f"  {word:<8} {v[0]:>6.2f} {v[1]:>6.2f} {v[2]:>6.2f} {v[3]:>6.2f}")

print()
print("  In real models (768+ dimensions), no single dimension has a clear meaning.")
print("  But GROUPS of dimensions together encode concepts like:")
print("    - Is it alive? (cat, dog = yes; car, wheel = no)")
print("    - Is it a vehicle? (car, engine, wheel = yes)")
print("    - Is it a pet? (cat, dog = yes)")
print()
print("  The model discovers these features on its own from context patterns.")
print("  Nobody told it 'cat is alive' — it learned that from sentences.")

---
## Question 2: Do we always need negative pairs?

Three approaches to training embeddings. Each handles negatives differently.
For each approach, we show: how it works, what the training data looks like, and a real example.

### Approach A: Explicit Contrastive (you provide negatives)

```
Positive: ("cat", "kitten")       -> pull together
Negative: ("cat", "airplane")     -> push apart
```

You must provide both. Used in older models (Word2Vec with negative sampling).

In [ ]:
print("Approach A: Explicit Contrastive Learning\n")

np.random.seed(10)
words_a = {w: np.random.randn(3) * 0.5 for w in ['cat', 'kitten', 'puppy', 'airplane', 'jet']}

positives = [('cat', 'kitten'), ('airplane', 'jet')]
negatives = [('cat', 'airplane'), ('kitten', 'jet')]

print("  You give the model:")
print("    Positives (similar):  cat <-> kitten,  airplane <-> jet")
print("    Negatives (different): cat <-> airplane, kitten <-> jet")
print()

for epoch in range(300):
    for w1, w2 in positives:
        diff = words_a[w2] - words_a[w1]
        words_a[w1] += 0.01 * diff
        words_a[w2] -= 0.01 * diff
    for w1, w2 in negatives:
        diff = words_a[w2] - words_a[w1]
        words_a[w1] -= 0.005 * diff
        words_a[w2] += 0.005 * diff

print("  After training:")
print(f"    cosine(cat, kitten)   = {cosine(words_a['cat'], words_a['kitten']):.3f}  (positive pair -> close)")
print(f"    cosine(airplane, jet) = {cosine(words_a['airplane'], words_a['jet']):.3f}  (positive pair -> close)")
print(f"    cosine(cat, airplane) = {cosine(words_a['cat'], words_a['airplane']):.3f}  (negative pair -> far)")
print()
print("  Problem: you need to manually create negative pairs.")
print("  With 1M positives, you need millions of negatives too.")

In [ ]:
print("Real-World Training Data for Approach A:\n")

print("  You prepare a CSV/JSON file like this:\n")
print("    ┌────────────────────────────────────┬───────────────────────────────────┬───────┐")
print("    | text_a                             | text_b                            | label |")
print("    |------------------------------------+-----------------------------------+-------|")
print("    | 'how to change a tire'             | 'tire replacement guide'          |   1   |")
print("    | 'how to change a tire'             | 'best chocolate cake recipe'      |   0   |")
print("    | 'symptoms of flu'                  | 'flu signs and treatment'         |   1   |")
print("    | 'symptoms of flu'                  | 'stock market predictions'        |   0   |")
print("    | 'python list comprehension'        | 'list comp syntax in python'      |   1   |")
print("    | 'python list comprehension'        | 'how to train a puppy'            |   0   |")
print("    └────────────────────────────────────┴───────────────────────────────────┴───────┘")
print("    label: 1 = similar (positive),  0 = different (negative)\n")

print("  Where does this data come from?")
print("    - Human annotators label pairs manually")
print("    - NLI datasets (Natural Language Inference): premise + entailment = positive")
print("    - You write a script to randomly pair unrelated sentences = negatives\n")

print("  Scale needed: ~100K-1M labeled pairs")
print("  Cost: expensive (human labeling)")
print("  Quality: high (humans verify each pair)")

### Approach B: Self-Supervised (NO negatives needed)

The model creates its own training signal from raw text. No pairs at all.

**Masked Language Model (BERT-style):**
```
Input:  "The [MASK] sat on the mat"
Target: "The  cat   sat on the mat"
```
The model must predict the hidden word from context.
To do this well, it must learn that 'cat' relates to 'sat', 'mat', etc.

In [ ]:
print("Approach B: Self-Supervised (Masked Language Model)\n")

print("  No pairs needed. Just raw text:\n")

raw_sentences = [
    "The cat sat on the mat",
    "A dog chased the cat",
    "My pet cat sleeps all day",
    "The car engine won't start",
    "Drive the car to the shop",
]

print("  Training examples (auto-generated from raw text):\n")
for sent in raw_sentences:
    words = sent.split()
    mask_idx = np.random.randint(0, len(words))
    masked_word = words[mask_idx]
    words_copy = words.copy()
    words_copy[mask_idx] = '[MASK]'
    print(f"    Input:  {' '.join(words_copy)}")
    print(f"    Target: predict '{masked_word}' from context")
    print()

print("  How this learns embeddings WITHOUT negatives:")
print("    - To predict [MASK] in 'The [MASK] sat on the mat'")
print("      the model must know 'cat' fits (animal that sits on mats)")
print("    - To predict [MASK] in 'The [MASK] engine won't start'")
print("      the model must know 'car' fits (thing with engines)")
print("    - The surrounding words ARE the implicit positive pairs")
print("    - The wrong predictions ARE the implicit negatives")

In [ ]:
print("Real-World Training Data for Approach B:\n")

print("  Source: Wikipedia, books, web crawls — just raw text, no labels\n")

print("  Literally just a text file:\n")
print("    ┌──────────────────────────────────────────────────────────────┐")
print("    | The Eiffel Tower is a wrought-iron lattice tower on the     |")
print("    | Champ de Mars in Paris. It is named after the engineer      |")
print("    | Gustave Eiffel, whose company designed and built the tower. |")
print("    |                                                             |")
print("    | Python is a high-level programming language. Its design     |")
print("    | philosophy emphasizes code readability with the use of      |")
print("    | significant indentation.                                    |")
print("    |                                                             |")
print("    | The mitochondria is the powerhouse of the cell. It         |")
print("    | generates most of the cell's supply of ATP.                 |")
print("    └──────────────────────────────────────────────────────────────┘")
print("    No labels. No pairs. Just text.\n")

print("  The model auto-generates training examples by masking:\n")
print("    Original:  'The Eiffel Tower is in Paris'")
print("    Masked:    'The [MASK] Tower is in Paris'    -> predict 'Eiffel'")
print("    Masked:    'The Eiffel Tower is in [MASK]'   -> predict 'Paris'")
print("    Masked:    'The Eiffel [MASK] is in Paris'   -> predict 'Tower'\n")

print("  BERT training data:")
print("    - English Wikipedia:       ~2.5 billion words")
print("    - BookCorpus:              ~800 million words")
print("    - Total:                   ~3.3 billion words")
print("    - Training time:           4 days on 64 TPUs")
print("    - Cost:                    ~$0 for data (it's all public text)")
print("    - Cost:                    ~$10K+ for compute")

---
### Deep Dive: What is BERT?

BERT = **B**idirectional **E**ncoder **R**epresentations from **T**ransformers (Google, 2018)

It's the model that made self-supervised embedding training practical.
Before BERT, most NLP models read text left-to-right (like GPT).
BERT reads in **both directions at once** — that's what "bidirectional" means.

In [ ]:
print("BERT vs GPT: Two different self-supervised approaches\n")

print("  GPT (left-to-right, like autocomplete):")
print("    Input:  'The cat sat on the'")
print("    Task:   predict next word -> 'mat'")
print("    It only sees words to the LEFT of the blank.\n")

print("  BERT (bidirectional, like fill-in-the-blank):")
print("    Input:  'The cat [MASK] on the mat'")
print("    Task:   predict masked word -> 'sat'")
print("    It sees words on BOTH sides of the blank.\n")

print("  Why bidirectional matters for embeddings:\n")
print("    Sentence: 'The bank by the river was steep'")
print("                                ^^^^^^^^^^^^")
print("    GPT sees:  'The bank' -> could be financial or riverbank")
print("    BERT sees: 'The bank by the river ... steep' -> definitely riverbank\n")

print("    BERT understands 'bank' means DIFFERENT things in:")
print("      'I went to the bank to deposit money'  -> financial")
print("      'I sat on the river bank'              -> ground")
print("    Because it reads the FULL context, not just the left side.")

In [ ]:
print("How BERT trains (step by step):\n")

print("  BERT has TWO training tasks:\n")

print("  Task 1: Masked Language Model (MLM) — 'fill in the blank'\n")
print("    Take a sentence, randomly mask 15% of words, predict them.\n")
original = "The quick brown fox jumps over the lazy dog"
words = original.split()
print(f"    Original:  {original}")
masks = [1, 4, 7]  # mask 'quick', 'jumps', 'lazy'
masked_words = words.copy()
for i in masks:
    masked_words[i] = '[MASK]'
print(f"    Masked:    {' '.join(masked_words)}")
print(f"    Targets:   predict 'quick', 'jumps', 'lazy'\n")

print("    The model gets EVERY word's context from both sides:")
print("      'The [MASK] brown fox' -> left='The', right='brown fox' -> 'quick'")
print("      'fox [MASK] over'      -> left='fox', right='over'      -> 'jumps'\n")

print("  Task 2: Next Sentence Prediction (NSP) — 'do these go together?'\n")
print("    Given two sentences, predict if sentence B follows sentence A.\n")
print("    Positive (B follows A):")
print("      A: 'The cat sat on the mat.'")
print("      B: 'It purred happily.'              -> YES, these are related\n")
print("    Negative (B is random):")
print("      A: 'The cat sat on the mat.'")
print("      B: 'Stock prices rose on Tuesday.'   -> NO, unrelated\n")

print("  Both tasks use ONLY raw text. No human labels needed.")
print("  The model generates its own training signal from the text itself.")

In [ ]:
print("BERT Architecture:\n")

print("  Input:  'The cat sat on the mat'")
print("           |    |   |   |   |   |")
print("          [tokenize into IDs]")
print("           |    |   |   |   |   |")
print("          [look up embedding for each token]  <- this is the embedding table")
print("           |    |   |   |   |   |")
print("          [transformer layer 1]  <- self-attention (each word looks at all others)")
print("           |    |   |   |   |   |")
print("          [transformer layer 2]")
print("           |    |   |   |   |   |")
print("          ... (12 layers for BERT-base, 24 for BERT-large)")
print("           |    |   |   |   |   |")
print("          [output: one vector per token, now context-aware]")
print("           |    |   |   |   |   |")
print("          v_The v_cat v_sat ...  <- each vector now 'knows' about ALL other words\n")

print("  The KEY difference from simple word embeddings:")
print("    Word2Vec:  'bank' always has the SAME vector")
print("    BERT:      'bank' has DIFFERENT vectors depending on context:\n")
print("      'river bank'  -> v_bank = [0.2, 0.8, -0.1, ...]  (nature)")
print("      'bank account'-> v_bank = [-0.3, 0.1, 0.9, ...]  (finance)\n")

print("  This is called CONTEXTUAL embeddings.")
print("  The transformer layers mix information between words,")
print("  so each word's vector is influenced by its neighbors.")

In [ ]:
print("BERT's role in the modern embedding pipeline:\n")

print("  BERT by itself is NOT a good embedding model for search.")
print("  It was designed for fill-in-the-blank, not for similarity.\n")

print("  But it's an excellent STARTING POINT. The modern pipeline:\n")

print("  Step 1: Pre-train BERT on raw text (self-supervised)")
print("    - 3.3 billion words from Wikipedia + books")
print("    - Learns language structure, word meanings, grammar")
print("    - Result: a model that UNDERSTANDS language\n")

print("  Step 2: Fine-tune for similarity (contrastive / in-batch negatives)")
print("    - Take pre-trained BERT")
print("    - Train with (query, document) pairs from search logs")
print("    - Now it produces vectors where similar = close\n")

print("  Step 3: Hard negative mining")
print("    - Find tricky pairs the model gets wrong")
print("    - 'car repair' vs 'car insurance' (similar words, different intent)")
print("    - Train more on these hard cases\n")

print("  Result: models like E5, BGE, OpenAI text-embedding-3\n")

print("  Timeline:")
print("    2013: Word2Vec        (one vector per word, no context)")
print("    2018: BERT            (contextual vectors, self-supervised)")
print("    2019: Sentence-BERT   (BERT fine-tuned for similarity)")
print("    2022: E5, BGE         (BERT + in-batch negatives + hard negatives)")
print("    2024: text-embedding-3 (proprietary, same general approach)")

---
### Approach C: In-Batch Negatives (free negatives)

The most popular approach for modern embedding models.
You only provide positive pairs. The other items in the same batch become negatives for free.

```
Batch of 4 positive pairs:
  ("cat", "kitten")        <- pair 0
  ("car", "vehicle")       <- pair 1
  ("python", "coding")     <- pair 2
  ("rain", "weather")      <- pair 3

For "cat":
  Positive: "kitten"                          (given)
  Negatives: "vehicle", "coding", "weather"   (FREE from the batch!)
```

In [ ]:
print("Approach C: In-Batch Negatives\n")

batch = [
    ("cat",    "kitten"),
    ("car",    "vehicle"),
    ("python", "coding"),
    ("rain",   "weather"),
]

print("  You provide ONLY positive pairs:")
for a, b in batch:
    print(f"    {a} <-> {b}")

print("\n  The model builds a similarity matrix:\n")
print(f"  {'':>10} {'kitten':>8} {'vehicle':>8} {'coding':>8} {'weather':>8}")
print(f"  {'':>10} {'--------':>8} {'--------':>8} {'--------':>8} {'--------':>8}")

ideal = [
    [0.95, 0.10, 0.05, 0.08],
    [0.12, 0.93, 0.08, 0.06],
    [0.04, 0.07, 0.91, 0.03],
    [0.06, 0.05, 0.04, 0.94],
]

queries = ['cat', 'car', 'python', 'rain']
for i, query in enumerate(queries):
    row = ''
    for j, val in enumerate(ideal[i]):
        marker = ' <--' if i == j else ''
        row += f"    {val:.2f}{marker}"
    print(f"  {query:>10}{row}")

print("\n  Diagonal = correct pairs (should be highest in each row)")
print("  Off-diagonal = in-batch negatives (should be low)")
print()
print("  With batch_size=4:  you get 3 free negatives per query")
print("  With batch_size=256: you get 255 free negatives per query")
print("  With batch_size=65536: you get 65535 free negatives (what OpenAI uses)")
print()
print("  This is why large batch sizes matter for embedding training.")
print("  More items in batch = more free negatives = better training.")

In [ ]:
print("Real-World Training Data for Approach C:\n")

print("  Source: search logs, Q&A sites, click data — only positive pairs\n")

print("  You prepare a CSV with ONLY positive pairs (no label column needed):\n")
print("    ┌────────────────────────────────────┬──────────────────────────────────────┐")
print("    | query                              | relevant_document                    |")
print("    |------------------------------------+--------------------------------------|")
print("    | 'how to reset iphone'              | 'Factory reset your iPhone: go to..' |")
print("    | 'python dict comprehension'        | 'Dict comprehensions create dicts..' |")
print("    | 'why is the sky blue'              | 'Rayleigh scattering causes short..' |")
print("    | 'back pain exercises'              | 'Stretches that help lower back..'   |")
print("    | 'convert pdf to word'              | 'Use Adobe Acrobat or free online..' |")
print("    | 'best noise canceling headphones'  | 'Sony WH-1000XM5 review: the best..'|")
print("    └────────────────────────────────────┴──────────────────────────────────────┘")
print("    No negatives! The batch provides them automatically.\n")

print("  Where does this data come from?")
print("    - Google/Bing search logs: (query, clicked result) = positive pair")
print("    - Stack Overflow: (question title, accepted answer) = positive pair")
print("    - Reddit: (post title, top comment) = positive pair")
print("    - Wikipedia: (section title, section content) = positive pair\n")

print("  Why this is the winning approach:")
print("    - Data is FREE (scraped from existing systems)")
print("    - No human labeling needed")
print("    - Scales to billions of pairs easily")
print("    - Negatives come free from the batch (batch=65536 -> 65535 negatives)")
print("    - This is how OpenAI, Cohere, and Google train their embedding models")

In [ ]:
print("Comparison of 3 Approaches:\n")

print(f"  {'Approach':<25} {'You Provide':<20} {'Negatives':<20} {'Used By'}")
print(f"  {'-'*25} {'-'*20} {'-'*20} {'-'*25}")
print(f"  {'A: Explicit Contrastive':<25} {'pos + neg pairs':<20} {'manual':<20} {'Word2Vec, early models'}")
print(f"  {'B: Self-Supervised':<25} {'raw text only':<20} {'implicit (wrong preds)':<20} {'BERT, GPT (pre-train)'}")
print(f"  {'C: In-Batch Negatives':<25} {'pos pairs only':<20} {'free from batch':<20} {'OpenAI, E5, BGE'}")
print()
print(f"  {'':25} {'Training Data':<20} {'Scale':<20} {'Cost'}")
print(f"  {'-'*25} {'-'*20} {'-'*20} {'-'*25}")
print(f"  {'A: Explicit Contrastive':<25} {'labeled CSV':<20} {'100K-1M pairs':<20} {'$$$ (human labels)'}")
print(f"  {'B: Self-Supervised':<25} {'raw text file':<20} {'billions of words':<20} {'free data, $$$ compute'}")
print(f"  {'C: In-Batch Negatives':<25} {'positive-only CSV':<20} {'10M-1B pairs':<20} {'free (scraped)'}")
print()
print("  Modern pipeline combines B then C:")
print("    Step 1: Pre-train with self-supervised (BERT) on raw text")
print("    Step 2: Fine-tune with in-batch negatives on curated pairs")
print("    Step 3: Hard negative mining (find tricky negatives to sharpen model)")

---
## Question 3: Embedding Model vs Vector DB

These are completely different things that work together.

In [ ]:
print("Embedding Model vs Vector Database\n")

print("  EMBEDDING MODEL                    VECTOR DATABASE")
print("  ========================           ========================")
print("  A neural network                   A database (like PostgreSQL)")
print("  Trained on billions of texts        Stores vectors you insert")
print("  Converts text -> vector             Searches vectors by similarity")
print("  Has NO memory of your data          Has ALL your data")
print("  Same output every time              Changes as you add/remove rows")
print("  ========================           ========================")
print()
print("  Analogy:")
print("    Model = a translator (converts language to numbers)")
print("    DB    = a filing cabinet (stores and retrieves those numbers)")
print()
print("  You need BOTH for RAG:")
print("    Model without DB = you can convert text to vectors, but can't search")
print("    DB without Model = you have a cabinet, but nothing to put in it")

In [ ]:
print("Full Pipeline: Model + DB Working Together\n")

np.random.seed(42)
fake_embeddings = {
    "Python is a programming language":     np.array([0.9, 0.1, -0.2]),
    "How to write Python code":             np.array([0.85, 0.15, -0.15]),
    "Best pasta recipe with tomato sauce":   np.array([-0.1, 0.9, 0.3]),
    "Italian cooking tips for beginners":    np.array([-0.05, 0.85, 0.35]),
    "Stock market crashed today":            np.array([0.1, -0.2, 0.9]),
}

def fake_embed(text):
    if text in fake_embeddings:
        return fake_embeddings[text]
    return np.array([0.8, 0.2, -0.1])

print("  STEP 1: Store documents in vector DB\n")
print("    For each document:")
print("      text -> [Embedding Model] -> vector -> [Vector DB]\n")

vector_db = []
for text in fake_embeddings:
    vec = fake_embed(text)
    vector_db.append((text, vec))
    print(f"    INSERT '{text[:40]}'")
    print(f"           -> [{', '.join(f'{v:.2f}' for v in vec)}]")

print(f"\n    Vector DB now has {len(vector_db)} rows.")

In [ ]:
print("  STEP 2: Search (query phase)\n")

query = "How do I learn Python?"
print(f"    User asks: '{query}'\n")

query_vec = fake_embed(query)
print(f"    Query -> [Embedding Model] -> [{', '.join(f'{v:.2f}' for v in query_vec)}]\n")

print(f"    [Vector DB] searches all rows by cosine similarity:\n")

results = []
for text, vec in vector_db:
    sim = cosine(query_vec, vec)
    results.append((sim, text))

results.sort(reverse=True)

print(f"    {'Score':>6}  Document")
print(f"    {'-----':>6}  {'--------'}")
for sim, text in results:
    marker = '  <-- most relevant' if sim == results[0][0] else ''
    print(f"    {sim:>6.3f}  {text}{marker}")

print(f"\n    Top result goes to the LLM as context for answering.")

In [ ]:
print("KEY INSIGHT: Model is frozen, DB is dynamic\n")

print("  The embedding model NEVER changes after training:")
print("    embed('cat') today  = [0.23, -0.41, 0.08, ...]")
print("    embed('cat') tomorrow = [0.23, -0.41, 0.08, ...]  <- exact same")
print()
print("  The vector DB changes constantly:")
print("    Monday:    500 documents stored")
print("    Tuesday:   INSERT 50 new docs -> 550 documents")
print("    Wednesday: DELETE 20 outdated docs -> 530 documents")
print()
print("  This is why you can:")
print("    - Add new documents to RAG without retraining the model")
print("    - Use the same model (OpenAI) for completely different databases")
print("    - Swap models without rebuilding your database (just re-embed)")
print()
print("  But if the model VERSION changes, you must re-embed everything.")
print("  Old vectors and new vectors live in different coordinate systems.")

---
## Summary

| Question | Answer |
|----------|--------|
| How does 'cat' get a vector? | Comparisons happened during training (billions of sentences). At query time it's just a lookup through the neural network. |
| Do we always need negatives? | No. Self-supervised (BERT) needs no pairs at all. In-batch negatives only need positives — the batch provides free negatives. |
| What is BERT? | A transformer that reads text bidirectionally. Pre-trained on raw text by predicting masked words. The foundation that modern embedding models are fine-tuned from. |
| Is embedding model = vector DB? | No. Model = translator (text to numbers, frozen). DB = storage (stores and searches, dynamic). If model version changes, re-embed everything. |

**See also:**
- `embedding_training.ipynb` — train a contrastive embedding model from scratch with code
- `vector_db_metrics.ipynb` — the 3 distance metrics (cosine, inner product, euclidean)
- `rag_example.ipynb` — full RAG pipeline using these components